# 01 · Meet LangGraph

> **Kernel:** `Phase 5 · LangGraph` · **Needs:** nothing for §3, Ollama for §4.

**LangGraph** is a framework for building AI systems as **graphs**: explicit
steps (nodes) connected by explicit transitions (edges), sharing an explicit
state. Its runtime runs that graph step by step, and it can **checkpoint**
after every step. That checkpoint is what enables pausing for a human,
surviving crashes, and replaying history.

This first notebook gives you the mental model, runs your first two graphs, and
maps out the rest of the section.

**You will learn**
- What LangGraph is, where it came from, and how it relates to LangChain
- The design philosophy: *low-level, explicit, you own the state*
- The four-step lifecycle every LangGraph program follows
- Your first graph (no model) and your first graph with a model

**Prerequisites:** Python and the idea of an LLM call. No earlier phase is
needed. *Optional deeper background:* `00_CONCEPTS/01` and `02` build the same
machinery by hand.

## 1 · Why LangGraph exists

A plain agent is a `while` loop: ask the model what to do, run the tool, repeat.
That loop has two hidden properties:

1. **The model decides the control flow.** You can't *guarantee* "tests run
   before commit". You can only ask nicely in the prompt.
2. **The work lives in memory.** If the process dies at step 19 of 20, you
   start again from step 1. A human can't approve something tomorrow, because
   the process would have to wait all night.

LangGraph's answer is to make the structure **explicit**:

| You declare… | …so the runtime can |
|---|---|
| a **state** schema | checkpoint it, resume it, show it to you at any step |
| **nodes** (named steps: code or LLM calls) | run, retry, time out, and test each step on its own |
| **edges** (fixed, conditional, parallel) | guarantee order, branch, and run branches concurrently |
| a **checkpointer** | pause for humans, survive crashes, time-travel through history |

The model still makes the judgement calls, inside nodes or in conditional edges
where *you* allow it. What you take back is the control flow you can't afford to
leave to chance.

## 2 · Where it comes from, and the philosophy

- **Made by LangChain Inc.**, open source (MIT). Version **1.0** shipped in
  October 2025; this section uses 1.2.
- **Inspired by Google's Pregel** (a system for large graph computation). The
  runtime runs in **supersteps**: all nodes that are due run in parallel, their
  updates are merged into the state, and the runtime works out what's due next.
  You'll see this in notebook 02.
- **Relationship to LangChain.** LangChain is the library of integrations
  (chat models, tools, retrievers). LangGraph is the *runtime* underneath. In
  LangChain 1.x, the high-level `create_agent` **is** a LangGraph graph. You'll
  use LangChain's chat-model interface for talking to models, but the
  orchestration is all LangGraph.

**The philosophy is "low-level on purpose".** LangGraph doesn't hide prompts,
invent agent roles, or decide your architecture. It gives you precise building
blocks and a durable runtime. You write more code than in a role-based framework
like CrewAI, and in exchange nothing happens that you didn't draw. That trade-off
is the main thing to remember when choosing between frameworks.

## 3 · The lifecycle, and a first graph (no model)

Every LangGraph program follows the same four steps:

```
 1. define the STATE      class State(TypedDict): ...
 2. build the GRAPH       builder = StateGraph(State)
                          builder.add_node(...) / builder.add_edge(...)
 3. COMPILE               graph = builder.compile(checkpointer=...)   ← validates the structure
 4. RUN                   graph.invoke(input) / graph.stream(input)
```

`StateGraph` is a **builder**: it only describes the graph. `compile()` checks
the structure and returns a runnable graph. Let's build the smallest useful
one: two nodes, one after the other.

In [ ]:
import sys, pathlib
SECTION = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
               if (p / 'common.py').exists())          # the 01_LANGGRAPH folder
sys.path.insert(0, str(SECTION))

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. State: the data every node can read. A TypedDict is just a typed dict.
class State(TypedDict):
    name: str
    greeting: str


# 2. Nodes: plain functions. They receive the state and return an UPDATE
#    (a dict of only the keys they change), never the whole state.
def greet(state: State) -> dict:
    return {'greeting': f"Hello, {state['name']}"}

def shout(state: State) -> dict:
    return {'greeting': state['greeting'].upper() + '!'}


builder = StateGraph(State)
builder.add_node('greet', greet)
builder.add_node('shout', shout)
builder.add_edge(START, 'greet')        # START and END are special built-in nodes
builder.add_edge('greet', 'shout')
builder.add_edge('shout', END)

# 3. Compile
graph = builder.compile()

# 4. Run
print(graph.invoke({'name': 'Ada'}))

What happened, step by step:

1. `invoke({'name': 'Ada'})` wrote the input into the state.
2. The edge from `START` made `greet` due. It returned `{'greeting': 'Hello, Ada'}`,
   and the runtime **merged** that into the state.
3. The edge `greet → shout` made `shout` due. Its update overwrote `greeting`.
4. `shout → END`: nothing more is due, so `invoke` returns the final state.

Notice that the nodes never touched each other. They communicate **only through
the state**. That's what makes each one independently testable:

In [ ]:
# A node is just a function: test it with a hand-made state, no graph needed.
assert greet({'name': 'Bo', 'greeting': ''}) == {'greeting': 'Hello, Bo'}

# The compiled graph can draw itself, which is handy for checking what you built.
print(graph.get_graph().draw_ascii())

`compile()` is also where structural mistakes are caught, *before* anything
runs. Here's an edge to a node that doesn't exist:

In [ ]:
broken = StateGraph(State)
broken.add_node('greet', greet)
broken.add_edge(START, 'greet')
broken.add_edge('greet', 'shuot')        # typo
try:
    broken.compile()
except ValueError as err:
    print('compile() refused:', err)

## 4 · A first graph with a model

Now a node that calls an LLM. We use a **chat model** from LangChain. Notebook 03
explains chat models properly; for now, `get_model()` (from this section's
`common.py`) returns a local Ollama model, and `model.invoke(text)` returns an
`AIMessage` whose `.content` is the reply.

The graph: `draft` asks the model for a one-line product idea, then `critique`
asks the model to criticise it. It's a fixed two-step chain, and its order is
guaranteed by the edge, not by a prompt.

In [ ]:
from common import get_model

model = get_model()          # ChatOllama(qwen3-coder:30b) unless your .env says otherwise


class IdeaState(TypedDict):
    topic: str
    idea: str
    critique: str


def draft(state: IdeaState) -> dict:
    reply = model.invoke(f"One-sentence startup idea about {state['topic']}. Sentence only.")
    return {'idea': reply.content}

def critique(state: IdeaState) -> dict:
    reply = model.invoke(f"In one sentence, the biggest risk of this idea: {state['idea']}")
    return {'critique': reply.content}


builder = StateGraph(IdeaState)
builder.add_node('draft', draft)
builder.add_node('critique', critique)
builder.add_edge(START, 'draft')
builder.add_edge('draft', 'critique')
builder.add_edge('critique', END)
idea_graph = builder.compile()

result = idea_graph.invoke({'topic': 'sidewalk delivery robots'})
print('IDEA    :', result['idea'])
print('CRITIQUE:', result['critique'])

Same four steps. The model is just something a node *uses*. LangGraph neither
knows nor cares what happens inside a node, which is why a node can be a plain
function, an LLM call, a tool call, or a whole agent.

You can also watch a run **step by step** instead of waiting for the end.
`stream(..., stream_mode='updates')` yields each node's update as it finishes.
Notebook 09 covers streaming in depth.

In [ ]:
for update in idea_graph.stream({'topic': 'solar-powered backpacks'}, stream_mode='updates'):
    for node, changes in update.items():
        print(f'[{node}] {changes}')

## 5 · What this section covers

Every notebook follows the same order: why, how it works, annotated code,
trade-offs and mistakes, and check yourself.

| # | Notebook | You'll be able to… |
|---|---|---|
| 01 | Meet LangGraph | explain what it is and run a graph (this notebook) |
| 02 | State, nodes, edges | design state with reducers; route with conditional edges |
| 03 | Chat models and messages | use Ollama / OpenAI / Anthropic interchangeably; messages; structured output |
| 04 | Tools and the ReAct agent | give a model tools; build an agent loop as a graph; use `create_agent` |
| 05 | Control flow | branch, loop with guards, fan out in parallel, map-reduce with `Send`, `Command` |
| 06 | Multi-agent systems | supervisor and handoff architectures |
| 07 | Human-in-the-loop | pause with `interrupt()`, approve/edit/reject, resume |
| 08 | Persistence and time travel | SQLite checkpoints, threads, crash recovery, replay and fork |
| 09 | Streaming and debugging | stream tokens and progress; inspect and trace runs |
| 10 | Subgraphs | compose graphs from graphs |
| 11 | MCP tools | connect graphs to MCP servers |
| 12 | Production concerns | retries, timeouts, error handling, limits, testing |
| ⭐ | Project | a research-and-report assistant that uses all of the above |

## 6 · When LangGraph is (and isn't) the right choice

| Choose LangGraph when… | Consider something else when… |
|---|---|
| You need **precise control** over steps, order and branching | You want a team of role-played agents with minimal code (CrewAI) |
| Runs are **long** and must **survive crashes** or **wait for humans** | A single agent with a few tools is enough (OpenAI Agents SDK, or `create_agent`) |
| You want to **inspect and replay** every step | You need multi-day, cross-service durable workflows at scale (Temporal) |
| You like explicit code over conventions | You want the framework to decide the architecture |

The comparison section of Phase 5 builds the same project in every framework,
so you can judge these trade-offs on real code.

## 7 · Common mistakes (beginner edition)

- **Returning the whole state from a node** "to be safe". Return only what
  changed; notebook 02 shows why this matters once branches run in parallel.
- **Mutating `state` in place** (`state['x'].append(...)`). The runtime may not
  see the change, and checkpoints won't record it. Always return an update.
- **Forgetting `compile()`**. A `StateGraph` is only a builder and has no `invoke`.
- **Forgetting edges to `END`**. A node with no outgoing edge simply ends that
  path, which is fine, but be deliberate about it.

## 8 · Check yourself

<details><summary><b>1.</b> What's the difference between <code>StateGraph</code> and what <code>compile()</code> returns?</summary>

`StateGraph` is a builder: it records nodes, edges and the state schema. `compile()` validates that structure (e.g. no edges to unknown nodes), attaches options such as a checkpointer, and returns a runnable graph with `invoke`, `stream`, `get_state`, and so on.
</details>

<details><summary><b>2.</b> A node returns <code>{'greeting': 'hi'}</code>. What happens to the other keys in the state?</summary>

Nothing: they keep their values. A node's return value is an *update* that the runtime merges into the state, key by key. Only the keys it names change.
</details>

<details><summary><b>3.</b> How do two nodes pass data to each other?</summary>

Only through the state. One node writes a key, and a later node reads it. They never call each other, which is why each can be tested alone with a hand-made state.
</details>

<details><summary><b>4.</b> How are LangGraph and LangChain related?</summary>

LangChain provides integrations (chat models, tools, etc.). LangGraph is the orchestration runtime, and LangChain 1.x's `create_agent` is itself built on LangGraph. You can use LangGraph with any Python code in the nodes; LangChain's chat models are just a convenient way to call LLMs.
</details>

<details><summary><b>5.</b> Why does LangGraph describe itself as "low-level", and what does that cost you?</summary>

It gives you primitives (state, nodes, edges, checkpoints) and doesn't hide prompts or impose an architecture. You get full control and transparency, but you write more code than in higher-level, convention-based frameworks.
</details>

## Takeaway

A LangGraph program is **state + nodes + edges**, compiled into a runnable graph.
Nodes read the state and return updates, edges decide what runs next, and the
runtime executes the graph step by step, which is what later lets it checkpoint,
pause, and resume.

Next → `02_state_nodes_edges.ipynb`: reducers, parallel branches, conditional
edges, and how the superstep runtime really works.